In [ ]:
import torch
import torch.nn as nn
import torch.autograd as autograd
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import time
torch.set_default_dtype(torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

: 

In [ ]:
G = 1.0
m1, m2, m3 = 1.0, 1.0, 1.0
eps = 1e-3

In [ ]:
n_hidden = 128
n_layers = 4
activation = nn.Tanh
lr = 1e-3
adam_epochs = 2000      # reduce/increase depending on available time
lbfgs_iters = 200
collocation_points = 400  # for physics loss
data_frac = 0.6  # fraction of reference points used as observed data

In [ ]:
t0, t1 = 0.0, 2.0  # increase t1 for longer trajectories (may require more training)
n_ref = 800  # number of reference points from numerical integrator

In [ ]:
def three_body_ode(t, y):
    # y: [x1,y1,x2,y2,x3,y3, vx1,vy1,vx2,vy2,vx3,vy3]
    x1, y1, x2, y2, x3, y3 = y[0:6]
    vx1, vy1, vx2, vy2, vx3, vy3 = y[6:12]

    def accel(xi, yi, xj, yj, mj):
        dx = xj - xi
        dy = yj - yi
        r2 = dx*dx + dy*dy + eps*eps
        r3 = r2 * np.sqrt(r2)
        return G * mj * dx / r3, G * mj * dy / r3

    a1x_12, a1y_12 = accel(x1, y1, x2, y2, m2)
    a1x_13, a1y_13 = accel(x1, y1, x3, y3, m3)
    a1x = a1x_12 + a1x_13
    a1y = a1y_12 + a1y_13

    a2x_21, a2y_21 = accel(x2, y2, x1, y1, m1)
    a2x_23, a2y_23 = accel(x2, y2, x3, y3, m3)
    a2x = a2x_21 + a2x_23
    a2y = a2y_21 + a2y_23

    a3x_31, a3y_31 = accel(x3, y3, x1, y1, m1)
    a3x_32, a3y_32 = accel(x3, y3, x2, y2, m2)
    a3x = a3x_31 + a3x_32
    a3y = a3y_31 + a3y_32

    return np.array([vx1, vy1, vx2, vy2, vx3, vy3, a1x, a1y, a2x, a2y, a3x, a3y])

In [ ]:
# ---------- Reference initial conditions (you can change these) ----------
# Choose a set of initial positions and velocities (non-trivial configuration)
y0 = np.array([ -1.0, 0.0,   1.0, 0.0,    0.0, 0.5,   # x1,y1, x2,y2, x3,y3
                0.0, 0.2,   0.0, -0.15,  -0.1, 0.0 ]) # vx1,vy1, vx2,vy2, vx3,vy3

In [ ]:
# ---------- Run numerical integrator to get reference trajectories ----------
t_eval = np.linspace(t0, t1, n_ref)
sol = solve_ivp(three_body_ode, [t0, t1], y0, t_eval=t_eval, rtol=1e-9, atol=1e-9, method='RK45')

if not sol.success:
    raise RuntimeError("solve_ivp failed")

ref_t = sol.t.reshape(-1,1).astype(np.float32)
ref_pos = sol.y.T[:, :6].astype(np.float32)  # positions only: x1,y1,x2,y2,x3,y3

In [ ]:
# Split observed data and collocation points
n_data = int(data_frac * n_ref)
indices = np.linspace(0, n_ref-1, n_ref, dtype=int)
np.random.seed(42)
obs_idx = np.sort(np.random.choice(indices, size=n_data, replace=False))
colloc_idx = np.sort(np.random.choice(indices, size=collocation_points, replace=False))

t_obs = torch.tensor(ref_t[obs_idx], device=device)
pos_obs = torch.tensor(ref_pos[obs_idx], device=device)

t_colloc = torch.tensor(ref_t[colloc_idx], device=device)

In [ ]:
# Convert full test times for plotting later
t_test = torch.tensor(ref_t, device=device)

print("Reference data points:", n_ref)
print("Observed data points used for data loss:", n_data)
print("Collocation points for physics loss:", collocation_points)

In [ ]:
# ---------- PINN Model ----------
class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)

class PINN(nn.Module):
    def __init__(self, in_dim=1, out_dim=6, n_layers=4, n_hidden=128):
        super().__init__()
        self.B = nn.Parameter(torch.randn(in_dim, n_hidden) * 2.0, requires_grad=False)
        layers = []
        layers.append(nn.Linear(n_hidden, n_hidden))
        layers.append(Sine())
        for _ in range(n_layers-2):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(Sine())
        layers.append(nn.Linear(n_hidden, out_dim))
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, t):
        x = torch.sin(t @ self.B)
        return self.net(x)

model = PINN(n_layers=n_layers, n_hidden=n_hidden).to(device)

In [ ]:
# ---------- Loss functions ----------
def compute_physics_residual(model, t_in):
    t = t_in.clone().detach().requires_grad_(True).to(device)
    pred = model(t)
    x1 = pred[:,0:1]; y1 = pred[:,1:2]
    x2 = pred[:,2:3]; y2 = pred[:,3:4]
    x3 = pred[:,4:5]; y3 = pred[:,5:6]

    v1x = autograd.grad(x1, t, torch.ones_like(x1), create_graph=True)[0]
    v1y = autograd.grad(y1, t, torch.ones_like(y1), create_graph=True)[0]
    v2x = autograd.grad(x2, t, torch.ones_like(x2), create_graph=True)[0]
    v2y = autograd.grad(y2, t, torch.ones_like(y2), create_graph=True)[0]
    v3x = autograd.grad(x3, t, torch.ones_like(x3), create_graph=True)[0]
    v3y = autograd.grad(y3, t, torch.ones_like(y3), create_graph=True)[0]

    a1x = autograd.grad(v1x, t, torch.ones_like(v1x), create_graph=True)[0]
    a1y = autograd.grad(v1y, t, torch.ones_like(v1y), create_graph=True)[0]
    a2x = autograd.grad(v2x, t, torch.ones_like(v2x), create_graph=True)[0]
    a2y = autograd.grad(v2y, t, torch.ones_like(v2y), create_graph=True)[0]
    a3x = autograd.grad(v3x, t, torch.ones_like(v3x), create_graph=True)[0]
    a3y = autograd.grad(v3y, t, torch.ones_like(v3y), create_graph=True)[0]

    def grav_acc(xi, yi, xj, yj, mj):
        dx = xj - xi; dy = yj - yi
        r2 = dx**2 + dy**2 + eps**2
        r3 = r2 * torch.sqrt(r2)
        return G * mj * dx / r3, G * mj * dy / r3

    f1x_12, f1y_12 = grav_acc(x1, y1, x2, y2, m2)
    f1x_13, f1y_13 = grav_acc(x1, y1, x3, y3, m3)
    f1x = f1x_12 + f1x_13; f1y = f1y_12 + f1y_13

    f2x_21, f2y_21 = grav_acc(x2, y2, x1, y1, m1)
    f2x_23, f2y_23 = grav_acc(x2, y2, x3, y3, m3)
    f2x = f2x_21 + f2x_23; f2y = f2y_21 + f2y_23

    f3x_31, f3y_31 = grav_acc(x3, y3, x1, y1, m1)
    f3x_32, f3y_32 = grav_acc(x3, y3, x2, y2, m2)
    f3x = f3x_31 + f3x_32; f3y = f3y_31 + f3y_32

    res = torch.cat([a1x - f1x, a1y - f1y, a2x - f2x, a2y - f2y, a3x - f3x, a3y - f3y], dim=1)
    
    v1_sq = v1x**2 + v1y**2
    v2_sq = v2x**2 + v2y**2
    v3_sq = v3x**2 + v3y**2
    K = 0.5 * (m1*v1_sq + m2*v2_sq + m3*v3_sq)
    
    r12 = torch.sqrt((x2-x1)**2 + (y2-y1)**2 + eps**2)
    r13 = torch.sqrt((x3-x1)**2 + (y3-y1)**2 + eps**2)
    r23 = torch.sqrt((x3-x2)**2 + (y3-y2)**2 + eps**2)
    U = -G * (m1*m2/r12 + m1*m3/r13 + m2*m3/r23)
    
    E = K + U
    dE_dt = autograd.grad(E, t, torch.ones_like(E), create_graph=True)[0]
    
    return res, dE_dt

def physics_loss(model, t_colloc):
    res, dE_dt = compute_physics_residual(model, t_colloc)
    return (res**2).mean() + (dE_dt**2).mean() * 0.1  # scale energy loss

def data_loss(model, t_obs, pos_obs):
    pred = model(t_obs)
    return ((pred - pos_obs)**2).mean()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
start_time = time.time()
print("Starting Adam training...")
for epoch in range(1, adam_epochs+1):
    model.train()
    optimizer.zero_grad()
    loss_d = data_loss(model, t_obs, pos_obs)
    loss_p = physics_loss(model, t_colloc)
    loss = loss_d * 1.0 + loss_p * 1.0
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0 or epoch == 1:
        print(f"Adam Epoch {epoch}/{adam_epochs}  |  Loss_total={loss.item():.6e}  data={loss_d.item():.6e}  phys={loss_p.item():.6e}")

adam_time = time.time() - start_time
print(f"Adam finished in {adam_time:.1f}s")


def closure():
    optimizer_lbfgs.zero_grad()
    loss_d = data_loss(model, t_obs, pos_obs)
    loss_p = physics_loss(model, t_colloc)
    loss = loss_d * 1.0 + loss_p * 1.0
    loss.backward()
    return loss

optimizer_lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=lbfgs_iters, lr=1.0, line_search_fn='strong_wolfe')

for i in range(3):
    loss_before = closure().item()
    optimizer_lbfgs.step(closure)
    loss_after = closure().item()
    print(f"LBFGS iter group {i+1}  loss_before={loss_before:.6e}  loss_after={loss_after:.6e}")

In [ ]:
  # ---------- Evaluate on test times and plot ----------
model.eval()
with torch.no_grad():
    pred_test = model(t_test).cpu().numpy()

ref = ref_pos  # numpy (n_ref,6)
pred = pred_test  # numpy (n_ref,6)

x1_ref, y1_ref = ref[:,0], ref[:,1]
x2_ref, y2_ref = ref[:,2], ref[:,3]
x3_ref, y3_ref = ref[:,4], ref[:,5]

x1_pred, y1_pred = pred[:,0], pred[:,1]
x2_pred, y2_pred = pred[:,2], pred[:,3]
x3_pred, y3_pred = pred[:,4], pred[:,5]

plt.figure(figsize=(8,8))
plt.plot(x1_ref, y1_ref, label='Body1 (ref)')
plt.plot(x2_ref, y2_ref, label='Body2 (ref)')
plt.plot(x3_ref, y3_ref, label='Body3 (ref)')
plt.plot(x1_pred, y1_pred, '--', label='Body1 (pinn)')
plt.plot(x2_pred, y2_pred, '--', label='Body2 (pinn)')
plt.plot(x3_pred, y3_pred, '--', label='Body3 (pinn)')
plt.legend()
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Three-Body: Reference vs PINN')
plt.axis('equal')
plt.show()

In [ ]:
# Also plot error over time (L2 per time step)
err = np.linalg.norm(ref - pred, axis=1)  # per time
plt.figure(figsize=(8,4))
plt.plot(ref_t.flatten(), err)
plt.xlabel('t')
plt.ylabel('L2 error (per time)')
plt.title('Pointwise L2 Error over time')
plt.show()

print("Done.")